# Maksimum Adil Model Karşılaştırması

## Tasarım İlkeleri

| Model | Eğitim | Karar eşiği |
|---|---|---|
| **BEME (binary)** | Yok — kural tabanlı | score ≥ 0 → pos |
| **LR** | 5-dış / 3-iç nested CV | GridSearch (C, weight) |
| **SVM** | 5-dış / 3-iç nested CV | GridSearch (C, weight) |
| **AutoBEME Apocalypse** | 5-fold OOF, n_funds=12 | τ=0.45 (sabit, diğer veriden) |
| **AutoBEME Vanguard** | 5-fold OOF, n_funds=12 | τ=0.30 |
| **AutoBEME Sovereign** | 5-fold OOF, n_funds=12 | τ=0.65 |
| **BERTurk — off-shelf** | Yok | binary label map |
| **BERTurk — fine-tuned** | 5-fold OOF, 3 epoch | 0.5 |
| **TurkishBERTweet — off-shelf** | Yok | neutral→neg map |
| **TurkishBERTweet — fine-tuned** | 5-fold OOF, 3 epoch | 0.5 |

**Ana metrik:** Macro-F1 (sınıflar eşit ağırlık)  
**İstatistik:** Çiftler arası bootstrap 95% CI (tüm model çiftleri)  
**Veri:** `data/goldset_binary_v2.csv` (veya mevcut goldset_binary.csv)

**Not — BERT fine-tuning:** `FINETUNE_BERT = True` yap. ≥50 positive örnek olduğunda anlamlı.

In [ ]:
# ═══════════════════════════════════════
#  AYARLAR — buradan yönet
# ═══════════════════════════════════════
from pathlib import Path

# Goldset: v2 varsa onu kullan, yoksa v1
DATA_DIR = Path("data")
GOLDSET_CSV = (
    DATA_DIR / "goldset_binary_v2.csv"
    if (DATA_DIR / "goldset_binary_v2.csv").exists()
    else DATA_DIR / "goldset_binary.csv"
)

CLASS_ORDER  = ["negative", "positive"]
RANDOM_SEED  = 42
N_OUTER      = 5    # dış CV katman sayısı
N_INNER      = 3    # iç CV (hyperparameter search)
BOOT_ITER    = 2000 # bootstrap iterasyon

FINETUNE_BERT = False   # True → BERT OOF fine-tuning (yavaş, ≥50 pos önerilir)
BERT_EPOCHS   = 3
BERT_LR       = 2e-5
BERT_BATCH    = 8

print(f"Goldset: {GOLDSET_CSV}")
print(f"BERT fine-tuning: {'AÇIK' if FINETUNE_BERT else 'KAPALI'}")

## 0. Paket Kontrolü

In [ ]:
import importlib

def chk(pkg, nm=None):
    ok = importlib.util.find_spec(nm or pkg) is not None
    print(f"  {'✅' if ok else '❌'}  {pkg}")
    return ok

print("Paketler:")
ok_sklearn = chk("scikit-learn", "sklearn")
ok_beme    = chk("beme")
ok_tf      = chk("transformers")
ok_torch   = chk("torch")
ok_peft    = chk("peft")

BEME_OK  = ok_beme
BERT_OK  = ok_tf and ok_torch
TWEET_OK = ok_tf and ok_torch and ok_peft

if FINETUNE_BERT and not BERT_OK:
    print("⚠️  BERT fine-tuning için: pip install transformers torch")

## 1. Veri

In [ ]:
import pandas as pd
import numpy as np

gold = pd.read_csv(GOLDSET_CSV)
gold["manual_label"]           = gold["manual_label"].astype(str).str.strip()
gold["heuristic_label_binary"] = gold["heuristic_label_binary"].astype(str).str.strip()
gold["window_text"]            = gold["window_text"].fillna("").astype(str)
gold = gold[gold["manual_label"].isin(CLASS_ORDER)].reset_index(drop=True)

vc  = gold["manual_label"].value_counts().reindex(CLASS_ORDER)
neg = vc["negative"]
pos = vc["positive"]

print(f"Goldset: {len(gold)} örnek  ({GOLDSET_CSV.name})")
print(f"  negative: {neg}  ({neg/len(gold):.1%})")
print(f"  positive: {pos}  ({pos/len(gold):.1%})")
print()

if pos < 30:
    print("⚠️  Positive örnek az (<30). Annotation devam ettir.")
elif pos < 50:
    print("⚠️  BERT fine-tuning için sınırda (30-50). Dikkatli yorumla.")
else:
    print("✅ BERT fine-tuning için yeterli (≥50 positive).")

y_true  = gold["manual_label"].astype(str)
X_text  = gold["window_text"].values
y_int   = np.array([0 if l == "negative" else 1 for l in gold["manual_label"]])
i2l     = {0: "negative", 1: "positive"}

## 2. Yardımcı Fonksiyonlar

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    precision_recall_fscore_support, confusion_matrix, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns

def metrics(y_true, y_pred, name=""):
    yt, yp = list(y_true), list(y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        yt, yp, labels=CLASS_ORDER, zero_division=0)
    yb = (np.array(yt) == "positive").astype(int)
    pb = (np.array(yp) == "positive").astype(int)
    try:
        auc = roc_auc_score(yb, pb)
    except Exception:
        auc = float("nan")
    row = {
        "model"       : name,
        "macro_f1"    : f1_score(yt, yp, labels=CLASS_ORDER, average="macro", zero_division=0),
        "cohen_kappa" : cohen_kappa_score(yt, yp, labels=CLASS_ORDER),
        "roc_auc"     : auc,
        "accuracy"    : accuracy_score(yt, yp),
        "weighted_f1" : f1_score(yt, yp, labels=CLASS_ORDER, average="weighted", zero_division=0),
    }
    for i, lbl in enumerate(CLASS_ORDER):
        row[f"{lbl}_f1"]        = f1[i]
        row[f"{lbl}_precision"] = p[i]
        row[f"{lbl}_recall"]    = r[i]
    return row

def plot_cm(yt, yp, title, ax):
    cm = confusion_matrix(yt, yp, labels=CLASS_ORDER, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
                ax=ax, vmin=0, vmax=1, linewidths=0.8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek")

all_metrics    = []   # tüm metrik dict'leri
all_preds      = {}   # model_name → pd.Series (oof predictions)
print("Hazır.")

## 3. BEME (Zero-Shot Baseline)

In [ ]:
y_beme = gold["heuristic_label_binary"].astype(str)
beme_m = metrics(y_true, y_beme, "BEME (binary)")
all_metrics.append(beme_m)
all_preds["BEME (binary)"] = y_beme

print(f"BEME: macro-F1={beme_m['macro_f1']:.3f}  "
      f"neg-F1={beme_m['negative_f1']:.3f}  pos-F1={beme_m['positive_f1']:.3f}  "
      f"kappa={beme_m['cohen_kappa']:.3f}")

## 4. Logistic Regression + SVM (Nested CV)

Genişletilmiş hyperparameter grid — önceki CV'den daha kapsamlı.

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV

def build_feats():
    return FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word", ngram_range=(1, 3),   # unigram→trigram
            min_df=1, lowercase=False, sublinear_tf=True)),
        ("char", TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5),
            min_df=1, lowercase=False, sublinear_tf=True)),
    ])

sel = FunctionTransformer(lambda df: df["window_text"], validate=False)

CONFIGS = {
    "Logistic Regression": (
        Pipeline([("sel", sel), ("feat", build_feats()),
                  ("clf", LogisticRegression(max_iter=5000, solver="lbfgs",
                                              random_state=RANDOM_SEED))]),
        {"clf__C": [0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0],
         "clf__class_weight": [None, "balanced"]}
    ),
    "Linear SVM": (
        Pipeline([("sel", sel), ("feat", build_feats()),
                  ("clf", LinearSVC(random_state=RANDOM_SEED))]),
        {"clf__C": [0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0],
         "clf__class_weight": [None, "balanced"]}
    ),
}

def nested_cv(pipe, pgrid, name):
    X = gold[["window_text"]].copy()
    y = gold["manual_label"].astype(str)
    outer = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_SEED)
    preds = np.empty(len(gold), dtype=object)
    for fold, (tr, te) in enumerate(outer.split(X, y), 1):
        gs = GridSearchCV(
            pipe, pgrid, scoring="f1_macro",
            cv=StratifiedKFold(N_INNER, shuffle=True, random_state=RANDOM_SEED + fold),
            n_jobs=-1, refit=True
        )
        gs.fit(X.iloc[tr], y.iloc[tr])
        preds[te] = gs.best_estimator_.predict(X.iloc[te])
        print(f"  fold {fold}/{N_OUTER}  best={gs.best_params_}")
    return pd.Series(preds, index=gold.index, dtype=str)

for mname, (pipe, pgrid) in CONFIGS.items():
    print(f"\n--- {mname} ---")
    oof = nested_cv(pipe, pgrid, mname)
    m   = metrics(y_true, oof, mname)
    all_metrics.append(m)
    all_preds[mname] = oof
    print(f"  macro-F1={m['macro_f1']:.3f}  neg-F1={m['negative_f1']:.3f}  "
          f"pos-F1={m['positive_f1']:.3f}  kappa={m['cohen_kappa']:.3f}")

print("\n✅ LR + SVM tamamlandı.")

## 5. AutoBEME (Apocalypse / Vanguard / Sovereign)

In [ ]:
if not BEME_OK:
    print("⚠️  beme yüklü değil → atlandı")
else:
    from beme import AutoBEME
    from sklearn.feature_extraction.text import TfidfVectorizer as TV

    MODES = {
        "AutoBEME Apocalypse": "balanced",
        "AutoBEME Vanguard"  : "recall",
        "AutoBEME Sovereign" : "precision",
    }
    kf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_SEED)

    for mname, mode in MODES.items():
        oof = np.empty(len(gold), dtype=object)
        print(f"\n--- {mname} ---")
        for fold, (tr, te) in enumerate(kf.split(X_text, y_int), 1):
            tv = TV(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, min_df=1)
            Xtr = tv.fit_transform(X_text[tr])
            Xte = tv.transform(X_text[te])
            mdl = AutoBEME(mode=mode, n_funds=12)
            mdl.fit(Xtr, y_int[tr])
            preds = mdl.predict(Xte)
            oof[te] = [i2l.get(int(p), "negative") for p in preds]
            print(f"  fold {fold}/{N_OUTER}", end=" ", flush=True)
        print()
        oof_s = pd.Series(oof, index=gold.index, dtype=str)
        m = metrics(y_true, oof_s, mname)
        all_metrics.append(m)
        all_preds[mname] = oof_s
        print(f"  macro-F1={m['macro_f1']:.3f}  neg-F1={m['negative_f1']:.3f}  "
              f"pos-F1={m['positive_f1']:.3f}  kappa={m['cohen_kappa']:.3f}")

    print("\n✅ AutoBEME tamamlandı.")

## 6. BERTurk (savasy) — Off-the-Shelf + Fine-Tuned

**Off-the-shelf:** Model indirilir, doğrudan tahmin yapılır.  
**Fine-tuned:** `FINETUNE_BERT = True` ise 5-fold OOF fine-tuning yapılır.

In [ ]:
SAVASY = "savasy/bert-base-turkish-sentiment-cased"

if not BERT_OK:
    print("⚠️  transformers/torch yüklü değil → atlandı")
else:
    try:
        from transformers import pipeline as hf_pipeline
        print(f"Yükleniyor: {SAVASY}")
        clf = hf_pipeline("text-classification", model=SAVASY,
                          tokenizer=SAVASY, device=-1, truncation=True)

        # Etiket kalibrasyonu
        cal = clf(["bu kararı destekliyoruz", "bu politika yanlış"], truncation=True)
        kw  = {"poz": "positive", "pos": "positive", "neg": "negative"}
        lmap = {o["label"]: next((v for k,v in kw.items() if k in o["label"].lower()),
                                  "negative")
                for o in cal}

        outs = clf(gold["window_text"].tolist(), truncation=True, batch_size=16)
        y_sav = pd.Series(
            [lmap.get(o["label"], "negative") for o in outs],
            index=gold.index, dtype=str)

        m = metrics(y_true, y_sav, "BERTurk (off-shelf)")
        all_metrics.append(m)
        all_preds["BERTurk (off-shelf)"] = y_sav
        print(f"macro-F1={m['macro_f1']:.3f}  neg-F1={m['negative_f1']:.3f}  "
              f"pos-F1={m['positive_f1']:.3f}  kappa={m['cohen_kappa']:.3f}")
        print("✅ BERTurk off-shelf tamamlandı.")

    except Exception as e:
        print(f"❌ {e}")

In [ ]:
if not FINETUNE_BERT:
    print("Fine-tuning atlandı (FINETUNE_BERT=False). Aktif etmek için config hücresini güncelle.")
elif not BERT_OK:
    print("⚠️  transformers/torch yüklü değil")
else:
    import torch
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        Trainer, TrainingArguments, DataCollatorWithPadding
    )
    from torch.utils.data import Dataset

    class SpeechDataset(Dataset):
        def __init__(self, enc, labels):
            self.enc    = enc
            self.labels = labels
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, i):
            return {k: v[i] for k, v in self.enc.items()} | {"labels": torch.tensor(self.labels[i])}

    LABEL2ID = {"negative": 0, "positive": 1}
    ID2LABEL = {0: "negative", 1: "positive"}

    tokenizer_sav = AutoTokenizer.from_pretrained(SAVASY)
    kf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_SEED)
    oof_ft = np.empty(len(gold), dtype=object)

    texts  = gold["window_text"].tolist()
    labels = [LABEL2ID[l] for l in gold["manual_label"]]

    for fold, (tr, te) in enumerate(kf.split(texts, labels), 1):
        print(f"\n--- BERTurk fine-tune fold {fold}/{N_OUTER} ---")
        enc_all = tokenizer_sav(
            [texts[i] for i in range(len(texts))],
            truncation=True, max_length=128, padding=True, return_tensors="pt")
        tr_enc = {k: v[list(tr)] for k, v in enc_all.items()}
        te_enc = {k: v[list(te)] for k, v in enc_all.items()}
        tr_ds  = SpeechDataset(tr_enc, [labels[i] for i in tr])
        te_ds  = SpeechDataset(te_enc, [labels[i] for i in te])

        model_ft = AutoModelForSequenceClassification.from_pretrained(
            SAVASY, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID,
            ignore_mismatched_sizes=True)

        args = TrainingArguments(
            output_dir=f"/tmp/berturk_ft_fold{fold}",
            num_train_epochs=BERT_EPOCHS,
            per_device_train_batch_size=BERT_BATCH,
            per_device_eval_batch_size=16,
            learning_rate=BERT_LR,
            warmup_ratio=0.1,
            weight_decay=0.01,
            logging_steps=999999,   # sessiz
            save_strategy="no",
            report_to="none",
        )
        trainer = Trainer(
            model=model_ft, args=args,
            train_dataset=tr_ds,
            data_collator=DataCollatorWithPadding(tokenizer_sav),
        )
        trainer.train()
        preds_raw = trainer.predict(te_ds).predictions
        pred_ids  = preds_raw.argmax(axis=-1)
        oof_ft[te] = [ID2LABEL[p] for p in pred_ids]
        print(f"  fold {fold} tamamlandı")

    oof_ft_s = pd.Series(oof_ft, index=gold.index, dtype=str)
    m = metrics(y_true, oof_ft_s, "BERTurk (fine-tuned)")
    all_metrics.append(m)
    all_preds["BERTurk (fine-tuned)"] = oof_ft_s
    print(f"\nmacro-F1={m['macro_f1']:.3f}  neg-F1={m['negative_f1']:.3f}  "
          f"pos-F1={m['positive_f1']:.3f}  kappa={m['cohen_kappa']:.3f}")
    print("✅ BERTurk fine-tuned tamamlandı.")

## 7. TurkishBERTweet — Off-the-Shelf + Fine-Tuned

In [ ]:
TWEET_MODEL    = "VRLLab/TurkishBERTweet-Lora-SA"
TWEET_ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
TWEET_BIN_MAP  = {"negative": "negative", "positive": "positive", "neutral": "negative"}

if not TWEET_OK:
    print("⚠️  peft yüklü değil → atlandı")
else:
    try:
        import torch
        from peft import PeftConfig, PeftModel
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        print(f"Yükleniyor: {TWEET_MODEL}")
        cfg = PeftConfig.from_pretrained(TWEET_MODEL)
        tok = AutoTokenizer.from_pretrained(cfg.base_model_name_or_path, padding_side="right")
        if tok.pad_token_id is None:
            tok.pad_token_id = tok.eos_token_id
        base = AutoModelForSequenceClassification.from_pretrained(
            cfg.base_model_name_or_path, return_dict=True, num_labels=3,
            id2label=TWEET_ID2LABEL, label2id={v:k for k,v in TWEET_ID2LABEL.items()})
        tw = PeftModel.from_pretrained(base, TWEET_MODEL)
        tw.eval()

        texts   = gold["window_text"].tolist()
        raw_ids = []
        with torch.no_grad():
            for i in range(0, len(texts), 16):
                enc = tok(texts[i:i+16], return_tensors="pt",
                          truncation=True, max_length=128, padding=True)
                raw_ids.extend(tw(**enc).logits.argmax(dim=-1).tolist())

        raw_labels = [TWEET_ID2LABEL[i] for i in raw_ids]
        y_tw = pd.Series(
            [TWEET_BIN_MAP[l] for l in raw_labels], index=gold.index, dtype=str)

        m = metrics(y_true, y_tw, "TurkishBERTweet (off-shelf)")
        all_metrics.append(m)
        all_preds["TurkishBERTweet (off-shelf)"] = y_tw
        print(f"Ham: {pd.Series(raw_labels).value_counts().to_dict()}")
        print(f"macro-F1={m['macro_f1']:.3f}  neg-F1={m['negative_f1']:.3f}  "
              f"pos-F1={m['positive_f1']:.3f}  kappa={m['cohen_kappa']:.3f}")
        print("✅ TurkishBERTweet off-shelf tamamlandı.")

    except Exception as e:
        print(f"❌ {e}")

In [ ]:
# TurkishBERTweet fine-tuning: base modeli (peft base) alıp binary olarak fine-tune
if not FINETUNE_BERT:
    print("Fine-tuning atlandı (FINETUNE_BERT=False).")
elif not TWEET_OK:
    print("⚠️  peft yüklü değil")
else:
    try:
        import torch
        from peft import PeftConfig
        from transformers import (
            AutoTokenizer, AutoModelForSequenceClassification,
            Trainer, TrainingArguments, DataCollatorWithPadding
        )

        LABEL2ID = {"negative": 0, "positive": 1}
        ID2LABEL  = {0: "negative", 1: "positive"}

        cfg_tw  = PeftConfig.from_pretrained(TWEET_MODEL)
        tok_tw  = AutoTokenizer.from_pretrained(
            cfg_tw.base_model_name_or_path, padding_side="right")
        if tok_tw.pad_token_id is None:
            tok_tw.pad_token_id = tok_tw.eos_token_id

        kf     = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_SEED)
        oof_tw = np.empty(len(gold), dtype=object)
        texts  = gold["window_text"].tolist()
        labels = [LABEL2ID[l] for l in gold["manual_label"]]

        for fold, (tr, te) in enumerate(kf.split(texts, labels), 1):
            print(f"\n--- TurkishBERTweet fine-tune fold {fold}/{N_OUTER} ---")
            enc_all = tok_tw(
                texts, truncation=True, max_length=128, padding=True, return_tensors="pt")
            tr_enc = {k: v[list(tr)] for k, v in enc_all.items()}
            te_enc = {k: v[list(te)] for k, v in enc_all.items()}
            tr_ds  = SpeechDataset(tr_enc, [labels[i] for i in tr])
            te_ds  = SpeechDataset(te_enc, [labels[i] for i in te])

            model_tw = AutoModelForSequenceClassification.from_pretrained(
                cfg_tw.base_model_name_or_path, num_labels=2,
                id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True)

            args_tw = TrainingArguments(
                output_dir=f"/tmp/tweet_ft_fold{fold}",
                num_train_epochs=BERT_EPOCHS,
                per_device_train_batch_size=BERT_BATCH,
                per_device_eval_batch_size=16,
                learning_rate=BERT_LR,
                warmup_ratio=0.1, weight_decay=0.01,
                logging_steps=999999, save_strategy="no", report_to="none",
            )
            trainer_tw = Trainer(
                model=model_tw, args=args_tw, train_dataset=tr_ds,
                data_collator=DataCollatorWithPadding(tok_tw),
            )
            trainer_tw.train()
            pr = trainer_tw.predict(te_ds).predictions.argmax(axis=-1)
            oof_tw[te] = [ID2LABEL[p] for p in pr]
            print(f"  fold {fold} tamamlandı")

        oof_tw_s = pd.Series(oof_tw, index=gold.index, dtype=str)
        m = metrics(y_true, oof_tw_s, "TurkishBERTweet (fine-tuned)")
        all_metrics.append(m)
        all_preds["TurkishBERTweet (fine-tuned)"] = oof_tw_s
        print(f"\nmacro-F1={m['macro_f1']:.3f}  neg-F1={m['negative_f1']:.3f}  "
              f"pos-F1={m['positive_f1']:.3f}  kappa={m['cohen_kappa']:.3f}")
        print("✅ TurkishBERTweet fine-tuned tamamlandı.")

    except Exception as e:
        print(f"❌ {e}")

## 8. Sonuç Tablosu

In [ ]:
summary = pd.DataFrame(all_metrics)
cols = ["model", "macro_f1", "cohen_kappa", "roc_auc",
        "negative_f1", "negative_precision", "negative_recall",
        "positive_f1", "positive_precision", "positive_recall",
        "accuracy", "weighted_f1"]
cols = [c for c in cols if c in summary.columns]
disp = summary[cols].sort_values("macro_f1", ascending=False).reset_index(drop=True)

display(disp.style
    .background_gradient(
        subset=[c for c in ["macro_f1","negative_f1","positive_f1","cohen_kappa"] if c in cols],
        cmap="RdYlGn", vmin=0, vmax=1)
    .format({c: "{:.3f}" for c in cols if c != "model"})
    .set_caption(f"Binary Model Karşılaştırma — {len(gold)} örnek — Macro-F1 sıralı"))

## 9. F1 Görsel

In [ ]:
plot_df = summary[["model","macro_f1","negative_f1","positive_f1"]].melt(
    id_vars="model", var_name="metric", value_name="score")
plot_df["metric"] = plot_df["metric"].map({
    "macro_f1"   : "Macro-F1",
    "negative_f1": "Negative F1 (saldırı)",
    "positive_f1": "Positive F1 (destek)",
})
order = summary.sort_values("macro_f1", ascending=False)["model"].tolist()

fig, ax = plt.subplots(figsize=(16, 5))
sns.barplot(data=plot_df, x="metric", y="score", hue="model",
            hue_order=order, palette="tab10", ax=ax)
ax.set_title(f"Maksimum Adil Karşılaştırma — {len(gold)} örnek",
             fontsize=13, fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("F1")
ax.set_ylim(0, 1.05)
ax.axhline(0.7, color="green", linestyle=":", alpha=0.5)
ax.legend(title="Model", bbox_to_anchor=(1.01,1), loc="upper left",
          frameon=False, fontsize=8)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig("fair_comparison_f1.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Confusion Matrix'ler

In [ ]:
n = len(all_preds)
cols_n = 4
rows_n = int(np.ceil(n / cols_n))
fig, axes = plt.subplots(rows_n, cols_n, figsize=(6*cols_n, 4.5*rows_n))
axes = np.atleast_1d(axes).reshape(rows_n, cols_n)
fig.suptitle("Confusion Matrices", fontsize=12, fontweight="bold")
for ax, (mname, preds) in zip(axes.flatten(), all_preds.items()):
    plot_cm(y_true, preds.astype(str), mname, ax)
for ax in axes.flatten()[n:]:
    ax.set_visible(False)
plt.tight_layout()
plt.savefig("fair_comparison_cms.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Bootstrap — TÜM Model Çiftleri (Pairwise 95% CI)

Sadece BEME'ye göre değil, tüm model çiftleri arasında istatistiksel fark testi.

In [ ]:
BOOT_ITER = 2000
rng = np.random.default_rng(RANDOM_SEED)
yt_r = y_true.reset_index(drop=True)

model_names = list(all_preds.keys())
n_models    = len(model_names)

# ── Pairwise bootstrap ──────────────────────────────────────────────────────
boot_rows = []
for ref_name in model_names:
    yr = all_preds[ref_name].astype(str).reset_index(drop=True)
    for cmp_name in model_names:
        if ref_name == cmp_name:
            continue
        yc = all_preds[cmp_name].astype(str).reset_index(drop=True)
        diffs = []
        for _ in range(BOOT_ITER):
            idx = rng.integers(0, len(yt_r), len(yt_r))
            yt, yri, yci = yt_r.iloc[idx], yr.iloc[idx], yc.iloc[idx]
            diffs.append(
                f1_score(yt, yci, labels=CLASS_ORDER, average="macro", zero_division=0) -
                f1_score(yt, yri, labels=CLASS_ORDER, average="macro", zero_division=0)
            )
        lo, hi = np.quantile(diffs, [0.025, 0.975])
        boot_rows.append({
            "baseline" : ref_name,
            "challenger": cmp_name,
            "mean_diff" : np.mean(diffs),
            "ci_lower"  : lo,
            "ci_upper"  : hi,
            "anlamli"   : "✅" if lo > 0 else ("⛔" if hi < 0 else "—"),
        })

boot_df = pd.DataFrame(boot_rows)

# ── Matris görselleştirmesi ─────────────────────────────────────────────────
pivot_mean = boot_df.pivot(index="baseline", columns="challenger", values="mean_diff")
pivot_sig  = boot_df.pivot(index="baseline", columns="challenger", values="anlamli")
pivot_mean = pivot_mean.reindex(index=model_names, columns=model_names)
pivot_sig  = pivot_sig.reindex(index=model_names, columns=model_names)

fig, ax = plt.subplots(figsize=(max(10, n_models*1.8), max(7, n_models*1.4)))
sns.heatmap(pivot_mean.fillna(0), annot=pivot_mean.applymap(lambda x: f"{x:+.3f}" if pd.notna(x) else ""),
            fmt="", cmap="RdYlGn", center=0, vmin=-0.3, vmax=0.3,
            linewidths=0.5, ax=ax)

# Anlamlı hücrelere border ekle
for i, ref in enumerate(model_names):
    for j, cmp in enumerate(model_names):
        if ref != cmp:
            val = pivot_sig.loc[ref, cmp] if cmp in pivot_sig.columns else None
            if val == "✅":
                ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                            edgecolor="black", lw=2))

ax.set_title(f"Pairwise Bootstrap 95% CI — macro-F1 farkı (satır baseline, sütun challenger)\n"
             f"Kalın border = anlamlı kazanç (CI_lower>0)  |  {BOOT_ITER} iter.",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Challenger →")
ax.set_ylabel("← Baseline")
plt.tight_layout()
plt.savefig("fair_comparison_pairwise_bootstrap.png", dpi=200, bbox_inches="tight")
plt.show()
print("Kaydedildi: fair_comparison_pairwise_bootstrap.png")

## 12. Karar ve Çıktı

In [ ]:
beme_f1 = summary[summary["model"]=="BEME (binary)"]["macro_f1"].values[0]
best = summary.sort_values("macro_f1", ascending=False).iloc[0]

print("=" * 70)
print(f"BEME baseline    : macro-F1={beme_f1:.3f}")
print(f"En iyi model     : {best['model']}")
print(f"  macro-F1 kazancı  : {best['macro_f1']-beme_f1:+.3f}")
print(f"  neg-F1            : {best['negative_f1']:.3f}")
print(f"  pos-F1            : {best['positive_f1']:.3f}")
print(f"  kappa             : {best['cohen_kappa']:.3f}")
print("=" * 70)

gain = best["macro_f1"] - beme_f1
if gain >= 0.08:
    print(f"\nKARARIMIZ: {best['model']} ANA MODEL")
elif gain >= 0.03:
    print(f"\nKARARIMIZ: BEME ana kalır. {best['model']} robustness katmanı.")
else:
    print("\nKARARIMIZ: BEME yeterli.")

print("\n--- Tam sıralama ---")
for _, row in summary.sort_values("macro_f1", ascending=False).iterrows():
    # Bootstrap anlamlılığı vs BEME
    if not boot_df.empty:
        sig_row = boot_df[
            (boot_df["baseline"]=="BEME (binary)") &
            (boot_df["challenger"]==row["model"])
        ]
        sig = sig_row["anlamli"].values[0] if len(sig_row) else ""
    else:
        sig = ""
    print(f"  {row['model']:<35} macro-F1={row['macro_f1']:.3f}  "
          f"neg-F1={row['negative_f1']:.3f}  "
          f"pos-F1={row['positive_f1']:.3f}  "
          f"kappa={row['cohen_kappa']:.3f}  {sig}")

# Kaydet
summary.to_csv("fair_comparison_summary.csv", index=False)
boot_df.to_csv("fair_comparison_bootstrap.csv", index=False)
print("\nKaydedildi: fair_comparison_summary.csv, fair_comparison_bootstrap.csv")

---
## Metodoloji Notu (Paper için)

- **Kural tabanlı:** BEME zero-shot, eğitim verisi yok → baseline
- **ML modelleri:** LR, SVM 5-dış/3-iç nested CV, hyperparameter search dahil
- **AutoBEME:** AutoBEME parametreleri (τ, T, L) Memory Text/SONAR/SPAMBASE üzerinden belirlendi, goldset'ten bağımsız → geçerli karşılaştırma
- **BERT off-shelf:** Eğitim yok → dezavantajlı ama baseline olarak raporlanabilir
- **BERT fine-tuned:** Goldset ile eğitildi, diğerlerle eşit koşul
- **İstatistik:** Pairwise bootstrap 95% CI (2000 iter.) — klasik pair test yerine bootstrap dağılım farkı
- **Limitation:** N={len(gold)}, 79/21 dengesiz → positive-F1 CI geniş, büyük örneklemle teyit önerilir